# From-scratch DPO on UltraFeedback (`dpo_uf`)
Policy and frozen reference both start from the SFT `best` checkpoint. Attach the saved output of `train_sft.ipynb` as an input dataset before running.

## Setup
Set `GITHUB_USER` below. The repository is private, so add a Kaggle secret named `GITHUB_TOKEN` (a GitHub personal access token with read access) under *Add-ons → Secrets*. The token is passed as an HTTP header and is not written into the cloned repository.

In [ ]:
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "Small-Model_Alignment"
REPO_DIR = f"/kaggle/working/{REPO_NAME}"

if not os.path.isdir(REPO_DIR):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    credentials = base64.b64encode(f"{GITHUB_USER}:{token}".encode()).decode()
    clone_command = [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Basic {credentials}",
        "clone",
        "-q",
        f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
        REPO_DIR,
    ]
    result = subprocess.run(clone_command)
    if result.returncode != 0:
        raise SystemExit("git clone failed")

os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib "trl==1.16.0"
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi

In [ ]:
import datasets
import peft
import torch
import transformers
import trl

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
if torch.cuda.device_count() < 2:
    raise SystemExit("Expected two GPUs. Select the GPU T4 x2 accelerator.")

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

## Inputs
Set `SFT_OUTPUTS` to the `outputs` folder inside the attached SFT notebook output.

In [ ]:
SFT_OUTPUTS = "/kaggle/input/YOUR-SFT-NOTEBOOK-OUTPUT/Small-Model_Alignment/outputs"

for required in [f"{SFT_OUTPUTS}/sft/best", f"{SFT_OUTPUTS}/data"]:
    if not os.path.isdir(required):
        raise SystemExit(f"Missing input folder: {required}")
print(sorted(os.listdir(f"{SFT_OUTPUTS}/data")))

## Train `dpo_uf`
Policy on `cuda:0` (fp32 weights, fp16 autocast), frozen reference on `cuda:1` (fp32 weights, same fp16 autocast). Metrics are appended to `outputs/dpo_uf/metrics.jsonl` at every evaluation step.

In [ ]:
!python scripts/train_dpo.py --config configs/dpo_uf.yaml --sft_path {SFT_OUTPUTS}/sft/best --data_dir {SFT_OUTPUTS}/data --output_dir outputs/dpo_uf

In [ ]:
!python scripts/plot_metrics.py --metrics_path outputs/dpo_uf/metrics.jsonl --output_dir outputs/dpo_uf/plots --run_name dpo_uf

## Archive small files
Checkpoints are not zipped. Use *Save Version* afterwards so `best/` and `final/` are kept in the notebook output.

In [ ]:
!cd outputs && zip -q -r /kaggle/working/dpo_uf_run_outputs.zip dpo_uf/metrics.jsonl dpo_uf/config_used.json dpo_uf/data_stats.json dpo_uf/prompt_template.txt dpo_uf/train_log.txt dpo_uf/plots
!ls -lh /kaggle/working/dpo_uf_run_outputs.zip